In [3]:
%run ./config

Box(children=(Label(value='Environment'), Dropdown(options=('dev', 'prod'), value='dev')))

Box(children=(Label(value='Cluster ID (vacío = serverless)'), Text(value='')))

In [4]:
# Create bronze schema

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{CATALOG}.{BRONZE_SCHEMA}
""")

""


In [5]:
# Create silver schema

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS
{CATALOG}.{SILVER_SCHEMA}
""")

""


In [6]:
# Create a volume

if env_cfg["storage_type"] == "external":
    spark.sql(f"""
    CREATE EXTERNAL VOLUME IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}.landing
    LOCATION '{env_cfg["external_path"]}'
    """)
else:
    spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOG}.{BRONZE_SCHEMA}.landing
    """)

In [7]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{BRONZE_MENU_FULL_TABLE}
(
    menu_item_id INT,
    item_name STRING,
    category STRING,
    price DOUBLE,
    _source_file STRING,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
""")

""


In [8]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{BRONZE_ORDER_FULL_TABLE}
(
    raw_json STRING,
    topic STRING,
    partition INT,
    offset LONG,
    timestamp TIMESTAMP,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
""")

""


In [9]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{SILVER_ORDER_FULL_TABLE}
(
    order_id INT,
    item_id INT,
    event_timestamp TIMESTAMP,
    topic STRING,
    partition INT,
    offset LONG,
    _ingestion_timestamp TIMESTAMP
)
USING DELTA
""")

""
